# Events dataset — ServiceNow

Lê `assets/incidents.csv` (base original da Locaweb, anonimizada) e gera dois CSVs no
formato da ServiceNow Table API (`GET /api/now/table/incident`, campos planos), com as
datas deslocadas para parecer um sistema rodando hoje:

- **legado**: dados reais de dez/2024–ago/2025, deslocados +12 meses → dez/2025–ago/2026
- **live**: dados reais de set/2025, deslocados +12 meses → só ontem e hoje (nada desde o recorte anterior, para não republicar o dia 18 aberto)

`categoria`/`subcategoria`/`produto` na base original estão ~63% nulos e, mesmo
preenchidos, são códigos anonimizados sem significado (`cat71`, `lhco`). São
reconstruídos por palavra-chave em `descricao_resumida`, agrupada por `item_configuracao`
— o mesmo CI sempre cai na mesma classificação.

No fim, cada linha vira um `POST /webhook/locaweb/service_now` com o body da Table API
plano — o mesmo shape que `set_mapping` no MCP espera. Configuração da origem não
passa daqui: skill `seed-origin-config`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().parent
CSV_PATH = REPO_ROOT / "assets" / "incidents.csv"

SEED = 42

df = pd.read_csv(CSV_PATH)
for col in ["aberto_em", "resolvido_em", "encerrado_em"]:
    df[col] = pd.to_datetime(df[col])

df.shape

(122543, 28)

## Recorte: legado (dez/24–ago/25) e live (set/25)

Fora dessa janela a base é ruído (2023 e a maior parte de 2024 têm dezenas de linhas/mês,
sem volume real) ou está fora do período que interessa (out/25 em diante, ignorado).

In [2]:
legacy_mask = (df["aberto_em"] >= "2024-12-01") & (df["aberto_em"] < "2025-09-01")
live_mask = (df["aberto_em"] >= "2025-09-01") & (df["aberto_em"] < "2025-10-01")

df_legacy = df[legacy_mask].copy()
df_live = df[live_mask].copy()

len(df_legacy), len(df_live)

(28659, 21561)

## Desloca as datas +12 meses

`live` usa o mês de set/2025 no original — deslocado, cai em set/2026. Fica só a janela
de ontem 00:00 até o fim de hoje, para não republicar o que já foi ingerido até o dia 18
como incidente ainda aberto.

In [3]:
TODAY = pd.Timestamp.now().normalize() + pd.Timedelta(hours=23, minutes=59, seconds=59)
YESTERDAY = pd.Timestamp.now().normalize() - pd.Timedelta(days=1)
SHIFT = pd.DateOffset(months=12)

def shift_dates(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    for col in ["aberto_em", "resolvido_em", "encerrado_em"]:
        frame[col] = frame[col] + SHIFT
    return frame

df_legacy = shift_dates(df_legacy)
df_live = shift_dates(df_live)

# Live simula operação em andamento: só ontem e hoje; nada aberto/resolvido/encerrado no futuro.
df_live = df_live[(df_live["aberto_em"] >= YESTERDAY) & (df_live["aberto_em"] <= TODAY)].copy()
for col in ["resolvido_em", "encerrado_em"]:
    future = df_live[col] > TODAY
    df_live.loc[future, col] = pd.NaT

df_legacy["aberto_em"].min(), df_legacy["aberto_em"].max(), df_live["aberto_em"].min(), df_live["aberto_em"].max()

(Timestamp('2025-12-02 15:48:48'),
 Timestamp('2026-08-31 23:59:40'),
 Timestamp('2026-09-01 00:01:52'),
 Timestamp('2026-09-21 23:58:26'))

## Category / subcategory / product sintéticos

Classificado por palavra-chave na `descricao_resumida` mais frequente de cada
`item_configuracao`, para ficar consistente por CI. `numero` continua a identidade
externa original — não é regerado.

In [4]:
# (product, category, subcategory) — sorteado por CI dentro do balde que a keyword aponta.
BUCKETS = {
    "monitoring": [
        ("Application Monitoring", "Software", "Monitoring Alert"),
        ("Application Monitoring", "Software", "Health Check Failure"),
    ],
    "connectivity": [
        ("Network", "Network", "Connectivity Loss"),
        ("Network", "Network", "Interface Flap"),
        ("Network", "Network", "High Latency"),
    ],
    "compute": [
        ("Cloud Servers", "Hardware", "CPU/Memory Threshold"),
        ("Cloud Servers", "Hardware", "Hypervisor Failure"),
    ],
    "storage": [
        ("Cloud Servers", "Hardware", "Disk/Swap Space"),
        ("Cloud Servers", "Hardware", "RAID/Storage Failure"),
    ],
    "backup": [
        ("Backup Service", "Software", "Backup Job Failure"),
    ],
    "database": [
        ("Database Hosting", "Software", "Replication Failure"),
        ("Database Hosting", "Software", "Database Service Down"),
    ],
    "service_check": [
        ("Shared Hosting", "Software", "Service Port Down"),
        ("Shared Hosting", "Software", "Certificate/TLS Issue"),
    ],
    "email": [
        ("Email Hosting", "Software", "Mailbox Access"),
        ("Email Hosting", "Software", "Mail Sync Failure"),
    ],
    "domains": [
        ("Domain Registration", "Software", "Domain Activation"),
        ("Website Builder", "Software", "Site Publishing"),
        ("Domain Registration", "Software", "DNS Zone Management"),
    ],
    "application": [
        ("VPS Hosting", "Software", "Application Crash"),
        ("VPS Hosting", "Software", "Service Unavailable"),
    ],
    "sales": [
        ("Shared Hosting", "Software", "Commercial Request"),
    ],
    "other": [
        ("Shared Hosting", "Software", "General Support"),
    ],
}

# Ordem importa: primeiro match vence, então padrões mais específicos vêm antes dos genéricos.
KEYWORDS = [
    ("monitoring", ["application monitoring", "alarm application monitoring"]),
    ("backup", ["backup", "bacula"]),
    (
        "database",
        [
            "postgresql", "mysql", "replication", "replica", "slave is not running", "slow quer",
            "redis", "proxysql", "galera", "mssql", "mirroring",
        ],
    ),
    (
        "service_check",
        [
            "check: http", "check: https", "check: ftp", "check: smtp", "check: mysql",
            "not running", "isn't running", "isn´t running", "isnt running",
            "certificate", "certificad", "varnish", "check local administrators",
            "admin service", "cert local", "check job error",
        ],
    ),
    (
        "storage",
        ["free disk", "free inodes", "espaço livre em disco", "raid", "storage", "swap", "dirty disk"],
    ),
    (
        "compute",
        [
            "cpu", "processor load", "memory is less", "memory free", "memory usage",
            "lack of available memory", "hypervisor", "iowait", "apache busy workers", "ntp",
        ],
    ),
    (
        "connectivity",
        [
            "icmp", "ping", "interface", "conectividade", "rede", "latenc", "tcp connection",
            "is down", "porta", "port", "link down", "bgp", "bandwidth", "pool:", "backends",
        ],
    ),
    (
        "email",
        ["email", "e-mail", "mailbox", "sincroniza", "smtp", "imap", "pop", "postfix", "pmta", "webmail", "smart host"],
    ),
    (
        "domains",
        ["domínio", "domain", "vhost", "ativação", "dns", "zona", "cpanel", "wordpress", "instalador"],
    ),
    (
        "application",
        [
            "gurnicorn", "aplicação", "application", "serviço", "web test", "nginx", "apache",
            "vm", "proxy", "exception", "iis", "cloudstack", "url not response", "servidor travado",
            "servidor indisponível", "fora do ar",
        ],
    ),
    ("sales", ["inside sales", "contratação", "cancelamento", "ajustar status", "upgrade"]),
]


def classify(description: str) -> str:
    text = str(description).lower()
    for bucket, keywords in KEYWORDS:
        if any(kw in text for kw in keywords):
            return bucket
    return "other"


top_description = (
    df.dropna(subset=["item_configuracao", "descricao_resumida"])
    .groupby("item_configuracao")["descricao_resumida"]
    .agg(lambda s: s.value_counts().idxmax())
)

ci_bucket = top_description.map(classify)

rng = np.random.default_rng(SEED)
all_cis = df["item_configuracao"].dropna().unique()
ci_bucket = ci_bucket.reindex(all_cis).fillna("other")

ci_classification = {}
for ci, bucket in ci_bucket.items():
    options = BUCKETS[bucket]
    product, category, subcategory = options[rng.integers(len(options))]
    ci_classification[ci] = {"u_product": product, "category": category, "subcategory": subcategory}

classification_df = pd.DataFrame.from_dict(ci_classification, orient="index")
classification_df.index.name = "item_configuracao"
ci_bucket.value_counts()

descricao_resumida
connectivity     2784
other            1750
backup           1223
storage           686
compute           571
service_check     567
application       417
email             329
database          304
domains           269
monitoring        200
sales              71
Name: count, dtype: int64

## Mapeamento para o vocabulário ServiceNow

In [5]:
PRIORITY = {1: "1", 2: "2", 3: "3", 4: "4", 5: "5"}

# ServiceNow incident.state: 1 New, 2 In Progress, 3 On Hold, 6 Resolved, 7 Closed, 8 Canceled.
# "Sem Intervenção" sempre tem encerrado_em preenchido e nunca codigo_fechamento — é um
# veredito de fechamento (close_code), não um estado de ciclo de vida à parte.
STATE = {
    "Encerrado": "7",
    "Encerrado Automaticamente": "7",
    "Sem Intervenção": "7",
    "Aguardando Problema": "3",
}

CLOSE_CODE = {
    "Falha de Aplicação": "Application Failure",
    "Falha de Sistema Operacional": "Operating System Failure",
    "Outro": "Other",
    "Falha causada pelo cliente": "Caused By Customer",
    "Sem retorno do solicitante": "No Caller Response",
    "Falha não reproduzida": "Not Reproducible",
    "Incidente causado por Change": "Caused By Change",
    "Falha de Cloud": "Cloud Failure",
    "Falha de Segurança": "Security Failure",
    "Falha de Hardware": "Hardware Failure",
    "Falso Positivo": "False Positive",
    "Falha de Monitoração": "Monitoring Failure",
    "Falha de Storage": "Storage Failure",
    "Falha de Banco de Dados": "Database Failure",
    "Falha de Redes": "Network Failure",
    "Gerado para auditoria de Segurança": "Security Audit",
    "Carta de Risco": "Risk Acceptance",
}
NO_INTERVENTION_CLOSE_CODE = "No Intervention Required"

OPENED_BY = {"Monitoramento": "monitoring.system", "Manual": "itsm.operator"}


def to_servicenow(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.merge(classification_df, on="item_configuracao", how="left")
    frame[["u_product", "category", "subcategory"]] = frame[["u_product", "category", "subcategory"]].fillna(
        {"u_product": "Shared Hosting", "category": "Software", "subcategory": "General Support"}
    )

    close_code = frame["codigo_fechamento"].map(CLOSE_CODE)
    close_code = close_code.where(frame["status"] != "Sem Intervenção", NO_INTERVENTION_CLOSE_CODE)

    out = pd.DataFrame(
        {
            "sys_id": frame["numero"],
            "number": frame["numero"],
            "short_description": frame["descricao_resumida"],
            "priority": frame["prioridade_codigo"].map(PRIORITY),
            "state": frame["status"].map(STATE),
            "opened_at": frame["aberto_em"].dt.strftime("%Y-%m-%d %H:%M:%S"),
            "resolved_at": frame["resolvido_em"].dt.strftime("%Y-%m-%d %H:%M:%S"),
            "closed_at": frame["encerrado_em"].dt.strftime("%Y-%m-%d %H:%M:%S"),
            "close_code": close_code,
            "close_notes": frame["solucao"],
            "assignment_group": frame["grupo_designado"],
            "cmdb_ci": frame["item_configuracao"],
            "opened_by": frame["aberto_por"].map(OPENED_BY),
            "category": frame["category"],
            "subcategory": frame["subcategory"],
            "u_product": frame["u_product"],
            "parent_incident": frame["incidente_pai"],
        }
    )
    return out


sn_legacy = to_servicenow(df_legacy)
sn_live = to_servicenow(df_live)

sn_legacy.head()

,sys_id,number,short_description,priority,state,opened_at,resolved_at,closed_at,close_code,close_notes,assignment_group,cmdb_ci,opened_by,category,subcategory,u_product,parent_incident
0,INC8520700,INC8520700,Problem: Free disk space is less than 10% on v...,4,7,2026-08-31 23:59:40,NaN,2026-09-01 03:13:55,No Intervention Required,NaN,Team14,IC00138,monitoring.system,Hardware,RAID/Storage Failure,Cloud Servers,NaN
1,INC8520698,INC8520698,Problem: Alarm Application Monitoring feeds Me...,4,7,2026-08-31 23:56:02,NaN,2026-09-01 00:29:26,No Intervention Required,NaN,Team14,IC04857,monitoring.system,Software,Health Check Failure,Application Monitoring,NaN
2,INC8520689,INC8520689,Problem: Sensor Umidade DC ITA00 - Corredor GH...,4,7,2026-08-31 23:48:49,NaN,2026-09-01 00:31:16,No Intervention Required,NaN,Team04,NaN,monitoring.system,Software,General Support,Shared Hosting,NaN
3,INC8520681,INC8520681,Problem: Check Application Monitoring,4,7,2026-08-31 23:39:19,NaN,2026-09-01 00:00:19,No Intervention Required,NaN,Team14,IC00012,monitoring.system,Software,Health Check Failure,Application Monitoring,NaN
4,INC8520667,INC8520667,Problem: IOwait grown up CPU queue,4,7,2026-08-31 23:21:05,NaN,2026-09-14 23:43:26,No Intervention Required,NaN,Team14,IC00220,monitoring.system,Software,General Support,Shared Hosting,NaN


In [6]:
sn_live.head()

,sys_id,number,short_description,priority,state,opened_at,resolved_at,closed_at,close_code,close_notes,assignment_group,cmdb_ci,opened_by,category,subcategory,u_product,parent_incident
0,INC8542134,INC8542134,Problem: Check Application Monitoring,4,7,2026-09-21 23:58:26,NaN,2026-09-21 23:58:54,No Intervention Required,NaN,Team14,IC02447,monitoring.system,Software,Health Check Failure,Application Monitoring,NaN
1,INC8542133,INC8542133,Problem: Check gmail_vmta Queue,3,7,2026-09-21 23:55:32,NaN,NaN,Caused By Customer,Definitiva,Team05,IC01328,monitoring.system,Software,Service Unavailable,VPS Hosting,INC8542128
2,INC8542132,INC8542132,Problem: Check gmail_vmta Queue,3,7,2026-09-21 23:55:22,NaN,NaN,Caused By Customer,Definitiva,Team05,IC01565,monitoring.system,Software,Service Port Down,Shared Hosting,INC8542128
3,INC8542131,INC8542131,Problem: Check gmail_vmta Queue,3,7,2026-09-21 23:55:09,NaN,NaN,Caused By Customer,Definitiva,Team05,IC01597,monitoring.system,Software,General Support,Shared Hosting,INC8542128
4,INC8542130,INC8542130,Problem: Check gmail_vmta Queue,3,7,2026-09-21 23:54:28,NaN,NaN,Caused By Customer,Definitiva,Team05,IC01487,monitoring.system,Software,Application Crash,VPS Hosting,INC8542128


## Exporta

In [7]:
OUT_LEGACY = REPO_ROOT / "assets" / "incidents_servicenow_legacy.csv"
OUT_LIVE = REPO_ROOT / "assets" / "incidents_servicenow_live.csv"

sn_legacy.to_csv(OUT_LEGACY, index=False)
sn_live.to_csv(OUT_LIVE, index=False)

len(sn_legacy), len(sn_live)

(28659, 14780)

## Envia para o webhook

O **legado** inteiro sobe em `POST /webhook/{tenant}/{source}/batch`: um JSON array de eventos da Table API, sem envelope. Cada request leva até `BATCH_SIZE` linhas (o máximo do gateway) e vira um produce no Kafka. `CONCURRENCY` é quantos batches ficam no ar juntos. O secret é o valor que o cadastro da origem devolveu.

In [ ]:
import asyncio
import hmac
import json
import os
from hashlib import sha256
from pathlib import Path

import httpx2
import numpy as np
import pandas as pd

_cwd = Path.cwd()
REPO_ROOT = _cwd if (_cwd / "assets").is_dir() else _cwd.parent
LIVE_CSV = REPO_ROOT / "assets" / "incidents_servicenow_live.csv"

sn_live = pd.read_csv(LIVE_CSV)
print(f"loaded {len(sn_live)} rows from {LIVE_CSV}", flush=True)

GATEWAY_URL = "http://gateway.204-168-208-210.sslip.io"
TENANT = "locaweb"
SOURCE = "service_now"
HMAC_SECRET = os.environ["WEBHOOK_SECRET"]
BATCH_SIZE = 500
CONCURRENCY = 4


def _jsonable(value):
    if value is None or pd.isna(value):
        return None
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if np.isnan(value) else float(value)
    if isinstance(value, str) and value in ("NaN", "NaT", "nan"):
        return None
    return value


def row_to_event(row: pd.Series) -> dict:
    return {key: _jsonable(value) for key, value in row.items()}


def _sign(secret: str, body: bytes) -> str:
    return "sha256=" + hmac.new(secret.encode(), body, sha256).hexdigest()


async def post_events(frame: pd.DataFrame) -> None:
    ordered = frame.sort_values("opened_at", na_position="last")
    events = [row_to_event(row) for _, row in ordered.iterrows()]
    total = len(events)
    path = f"/webhook/{TENANT}/{SOURCE}/batch"
    batches = [events[start : start + BATCH_SIZE] for start in range(0, total, BATCH_SIZE)]
    sem = asyncio.Semaphore(CONCURRENCY)
    done = 0
    lock = asyncio.Lock()

    async def send(client: httpx2.AsyncClient, batch: list[dict]) -> None:
        nonlocal done
        body = json.dumps(batch, ensure_ascii=False, allow_nan=False).encode()
        headers = {
            "content-type": "application/json",
            "X-Signature": _sign(HMAC_SECRET, body),
        }
        async with sem:
            resp = await client.post(path, content=body, headers=headers)
            resp.raise_for_status()
        async with lock:
            done += len(batch)
            n = done
        if n == len(batch) or n % (BATCH_SIZE * 4) == 0 or n == total:
            print(f"posted {n}/{total}", flush=True)

    async with httpx2.AsyncClient(base_url=GATEWAY_URL, timeout=60) as client:
        await asyncio.gather(*(send(client, batch) for batch in batches))


await post_events(sn_live)
